# Tính toán Selective Search Proposals cho Fast R-CNN
**Mục đích:** Tính trước (pre-compute) vùng đề xuất bằng Selective Search cho toàn bộ train / valid / test.

Notebook tự động phát hiện số nhân CPU, chạy **benchmark** để tìm số luồng tối ưu thực tế,
sau đó tính proposals với tốc độ nhanh nhất có thể trên máy hiện tại.

> **Yêu cầu:** `opencv-contrib-python` (không phải `opencv-python` thông thường)

## 1. Kiểm tra thư viện

In [ ]:
# Bỏ comment nếu chưa cài:
# !pip install opencv-contrib-python pycocotools tqdm psutil

import cv2, pickle
print('opencv  :', cv2.__version__)

try:
    cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
    print('SelectiveSearch : OK (opencv-contrib đã cài)')
except AttributeError:
    print('LỖI: Cần cài opencv-contrib-python!')
    print('  pip uninstall opencv-python && pip install opencv-contrib-python')

## 2. Import

In [ ]:
import os, time, pickle, multiprocessing, random
import cv2
import numpy as np
from pycocotools.coco import COCO
from tqdm.notebook import tqdm

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print('psutil không có — cài thêm để đọc physical cores chính xác hơn')

print('Import OK')

## 3. Phân tích CPU — Tự động tối ưu số luồng

Selective Search là tác vụ **CPU-bound thuần túy**: mỗi ảnh độc lập, không chia sẻ bộ nhớ.
Số luồng tối ưu thường nằm trong khoảng `physical_cores` đến `logical_cores - 1`:
- **Ít hơn** `physical_cores` → lãng phí tài nguyên
- **Nhiều hơn** `logical_cores` → tranh chấp core, tốc độ giảm
- **Trừ 1** logical core → nhường cho HĐH và Jupyter, tránh treo máy

In [ ]:
# ── Phát hiện CPU ────────────────────────────────────────────────────────────
logical_cores  = multiprocessing.cpu_count()
physical_cores = psutil.cpu_count(logical=False) if HAS_PSUTIL else logical_cores // 2
cpu_freq_mhz   = (psutil.cpu_freq().max if HAS_PSUTIL and psutil.cpu_freq() else 0)
ram_gb         = (psutil.virtual_memory().total / 1024**3 if HAS_PSUTIL else 0)

print(f'Physical cores : {physical_cores}')
print(f'Logical cores  : {logical_cores}  (HyperThreading: {"ON" if logical_cores > physical_cores else "OFF"})')
print(f'CPU max freq   : {cpu_freq_mhz:.0f} MHz' if cpu_freq_mhz else 'CPU max freq   : N/A')
print(f'RAM total      : {ram_gb:.1f} GB' if ram_gb else 'RAM total      : N/A')

# ── Tự động chọn số luồng tối đa ─────────────────────────────────────────────
# Dùng tất cả logical cores trừ 1 (để HĐH + Jupyter không bị nghẹt)
AUTO_CORES = max(1, logical_cores - 1)

print(f'\n→ Số luồng sẽ dùng: {AUTO_CORES} (có thể điều chỉnh sau khi benchmark)')

# ── Cấu hình đường dẫn ────────────────────────────────────────────────────────
BASE_DIR    = os.path.dirname(os.path.abspath('__file__'))
DATASET_DIR = os.path.join(BASE_DIR, 'dataset_coco_cropped')

TRAIN_IMG = os.path.join(DATASET_DIR, 'train', 'images')
TRAIN_ANN = os.path.join(DATASET_DIR, 'train', '_annotations.coco.json')
VALID_IMG = os.path.join(DATASET_DIR, 'valid', 'images')
VALID_ANN = os.path.join(DATASET_DIR, 'valid', '_annotations.coco.json')
TEST_IMG  = os.path.join(DATASET_DIR, 'test',  'images')
TEST_ANN  = os.path.join(DATASET_DIR, 'test',  '_annotations.coco.json')

TRAIN_SS_CACHE = os.path.join(DATASET_DIR, 'train_ss_proposals.pkl')
VALID_SS_CACHE = os.path.join(DATASET_DIR, 'valid_ss_proposals.pkl')
TEST_SS_CACHE  = os.path.join(DATASET_DIR, 'test_ss_proposals.pkl')

MAX_PROPOSALS = 2000
OVERWRITE     = False

print('\n===== PATHS =====')
for name, p in [('DATASET_DIR', DATASET_DIR),
                ('TRAIN_IMG', TRAIN_IMG), ('TRAIN_ANN', TRAIN_ANN),
                ('VALID_IMG', VALID_IMG), ('VALID_ANN', VALID_ANN),
                ('TEST_IMG',  TEST_IMG),  ('TEST_ANN',  TEST_ANN)]:
    status = 'OK' if os.path.exists(p) else 'KHÔNG TÌM THẤY'
    print(f'  {status:20}  {name}')

## 3b. Kiểm tra cache hiện có

Trước khi chạy benchmark và tính toán, kiểm tra xem file `.pkl` nào đã tồn tại.

- **File đã có + `OVERWRITE=False`** → bỏ qua, load trực tiếp (tiết kiệm thời gian)
- **File chưa có** → sẽ được tính mới
- **`OVERWRITE=True`** → tính lại tất cả dù file đã tồn tại

In [ ]:
import datetime

# ── Kiểm tra trạng thái các file cache ───────────────────────────────────────
CACHE_FILES = [
    ('Train',      TRAIN_SS_CACHE, TRAIN_ANN),
    ('Validation', VALID_SS_CACHE, VALID_ANN),
    ('Test',       TEST_SS_CACHE,  TEST_ANN),
]

print('=' * 65)
print('  KIỂM TRA CACHE SELECTIVE SEARCH PROPOSALS')
print('=' * 65)
print(f'  OVERWRITE = {OVERWRITE}')
print()

need_compute = []
all_exist    = True

for split_name, cache_path, ann_path in CACHE_FILES:
    if os.path.exists(cache_path):
        # Lấy thông tin file
        size_mb  = os.path.getsize(cache_path) / 1024**2
        mtime    = os.path.getmtime(cache_path)
        mod_time = datetime.datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')

        # Đọc nhanh để biết số ảnh trong cache
        try:
            with open(cache_path, 'rb') as f:
                _data = pickle.load(f)
            n_imgs = len(_data)
            del _data
            img_info = f'{n_imgs} ảnh'
        except Exception as e:
            img_info = f'lỗi đọc: {e}'

        status_icon = '✓' if not OVERWRITE else '↺'
        action_text = 'BỎ QUA (OVERWRITE=False)' if not OVERWRITE else 'SẼ TÍNH LẠI (OVERWRITE=True)'
        print(f'  [{status_icon}] {split_name:<12}  ĐÃ CÓ')
        print(f'       Kích thước : {size_mb:.1f} MB')
        print(f'       Cập nhật   : {mod_time}')
        print(f'       Nội dung   : {img_info}')
        print(f'       → {action_text}')
        if OVERWRITE:
            need_compute.append(split_name)
    else:
        all_exist = False
        need_compute.append(split_name)
        print(f'  [✗] {split_name:<12}  CHƯA CÓ — sẽ được tính')
    print()

# ── Tóm tắt hành động ─────────────────────────────────────────────────────────
print('─' * 65)
if not need_compute:
    print(f'   Tất cả 3 tập đã có cache — KHÔNG cần tính lại.')
    print(f'    Nếu muốn tính lại, đặt OVERWRITE = True ở cell trên.')
else:
    print(f'  → {len(need_compute)}/3 tập cần tính: {", ".join(need_compute)}')
print('=' * 65)

## 4. Worker function — Selective Search

Hàm phải đặt ở **module level** (ngoài cùng) để `multiprocessing` pickle được sang subprocess.
Resize ảnh đồng bộ với `GeneralizedRCNNTransform(min_size=640, max_size=1024)` trước khi chạy SS.

In [ ]:
def process_single_image(args):
    """
    Worker: đọc 1 ảnh → resize (đồng bộ model) → Selective Search Fast
    → trả tọa độ về không gian ảnh gốc.
    """
    img_id, img_path, max_props = args
    img = cv2.imread(img_path)
    if img is None:
        return img_id, []

    orig_h, orig_w = img.shape[:2]

    # Scale đồng bộ GeneralizedRCNNTransform(min_size=640, max_size=1024)
    scale   = min(640.0 / min(orig_h, orig_w), 1024.0 / max(orig_h, orig_w))
    new_h, new_w = int(orig_h * scale), int(orig_w * scale)
    img_r   = cv2.resize(img, (new_w, new_h))

    ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
    ss.setBaseImage(img_r)
    ss.switchToSelectiveSearchFast()
    rects = ss.process()[:max_props]

    boxes = []
    for (x, y, bw, bh) in rects:
        x1 = max(0.0,          float(x)       / scale)
        y1 = max(0.0,          float(y)       / scale)
        x2 = min(float(orig_w), float(x + bw) / scale)
        y2 = min(float(orig_h), float(y + bh) / scale)
        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
    return img_id, boxes


print('Worker function OK')

## 5. Benchmark — Tìm số luồng tối ưu thực tế

Chạy thử trên **20 ảnh mẫu** với nhiều mức core khác nhau để đo thời gian thực tế.
Selective Search đã có OpenMP bên trong nên oversubscription có thể xảy ra — benchmark sẽ xác nhận.

> Cell này **không ảnh hưởng** đến kết quả cuối — chỉ để chọn `NUM_CORES` tốt nhất.

In [ ]:
import matplotlib.pyplot as plt

# Lấy 20 ảnh mẫu từ train set
_coco_bm = COCO(TRAIN_ANN)
_all_ids  = list(_coco_bm.imgs.keys())
random.seed(42)
_sample_ids = random.sample(_all_ids, min(20, len(_all_ids)))
_bm_tasks   = [
    (iid, os.path.join(TRAIN_IMG, _coco_bm.imgs[iid]['file_name']), MAX_PROPOSALS)
    for iid in _sample_ids
]

# Danh sách core cần test: 1, 2, 4, physical, logical-1, logical
_test_cores = sorted(set(filter(lambda x: 1 <= x <= logical_cores, [
    1, 2, 4,
    physical_cores,
    max(1, logical_cores // 2),
    max(1, logical_cores - 1),
    logical_cores,
])))

print(f'Benchmark: {len(_bm_tasks)} ảnh × {len(_test_cores)} cấu hình core: {_test_cores}')
print('Đang chạy...\n')

_bm_results = {}   # {n_cores: time_per_img}
for n in _test_cores:
    chunksize = max(1, len(_bm_tasks) // (n * 2))
    t0 = time.time()
    with multiprocessing.Pool(processes=n, maxtasksperchild=200) as pool:
        _ = list(pool.imap_unordered(process_single_image, _bm_tasks, chunksize=chunksize))
    elapsed = time.time() - t0
    per_img = elapsed / len(_bm_tasks)
    _bm_results[n] = per_img
    print(f'  {n:>2} cores : tổng {elapsed:5.1f}s | {per_img:.3f}s/ảnh  | speedup vs 1-core: {_bm_results[1]/per_img:.1f}×')

# Chọn số core tối ưu (thời gian/ảnh nhỏ nhất)
NUM_CORES = min(_bm_results, key=_bm_results.get)
print(f'\n→ Tối ưu: NUM_CORES = {NUM_CORES} ({_bm_results[NUM_CORES]:.3f}s/ảnh)')

# Biểu đồ
fig, ax = plt.subplots(figsize=(8, 4))
cores_list   = list(_bm_results.keys())
speedup_list = [_bm_results[1] / _bm_results[n] for n in cores_list]
ideal_list   = [float(n) for n in cores_list]

ax.plot(cores_list, speedup_list, 'b-o', lw=2, ms=7, label='Thực tế')
ax.plot(cores_list, ideal_list,   'r--', lw=1, alpha=0.6, label='Lý tưởng (tuyến tính)')
ax.axvline(x=NUM_CORES, color='green', ls='--', lw=1.5,
           label=f'Tối ưu: {NUM_CORES} cores')
ax.set_xlabel('Số luồng (cores)')
ax.set_ylabel('Speedup so với 1 core')
ax.set_title('Benchmark Selective Search — Tìm số luồng tối ưu', fontweight='bold')
ax.legend(); ax.grid(True, ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(DATASET_DIR, 'ss_benchmark.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Thời gian ước tính toàn bộ 4915 ảnh: {4915 * _bm_results[NUM_CORES] / 60:.1f} phút')

## 6. Master function — Tính toàn bộ với cấu hình tối ưu

**Tối ưu so với multiprocessing thông thường:**
- `chunksize` tự động: giảm số lần IPC (giao tiếp giữa process), đặc biệt hiệu quả khi nhiều core
- `maxtasksperchild=200`: worker tự restart sau 200 task để giải phóng RAM bị SS tích lũy
- `imap_unordered`: không chờ thứ tự, lấy kết quả ngay khi worker xong

In [ ]:
def precompute_ss_proposals(img_dir, ann_file, cache_path,
                             max_props=2000, num_cores=None, overwrite=False):
    """
    Tính Selective Search proposals cho toàn bộ tập ảnh — multiprocessing tối ưu.

    Tối ưu kỹ thuật:
      - chunksize  = ceil(n_imgs / (num_cores * 4))  → giảm IPC overhead
      - maxtasksperchild = 200                       → giải phóng RAM worker
      - imap_unordered                               → không chờ thứ tự
    """
    if num_cores is None:
        num_cores = max(1, multiprocessing.cpu_count() - 1)

    if os.path.exists(cache_path) and not overwrite:
        print(f'  Cache đã có: {os.path.basename(cache_path)}')
        t0 = time.time()
        with open(cache_path, 'rb') as f:
            data = pickle.load(f)
        print(f'  Load xong: {len(data)} ảnh ({time.time()-t0:.1f}s)')
        return data

    coco    = COCO(ann_file)
    img_ids = list(coco.imgs.keys())
    tasks   = [
        (iid, os.path.join(img_dir, coco.imgs[iid]['file_name']), max_props)
        for iid in img_ids
    ]

    # chunksize: mỗi lần gửi cho worker bao nhiêu task
    # Nhiều task/lần → ít IPC → nhanh hơn khi num_cores lớn
    chunksize = max(1, len(tasks) // (num_cores * 4))

    print(f'  {len(img_ids)} ảnh | {num_cores} cores | chunksize={chunksize} | maxtasksperchild=200')
    t0 = time.time()

    proposals_dict = {}
    with multiprocessing.Pool(
            processes=num_cores,
            maxtasksperchild=200   # worker restart sau 200 task → giải phóng RAM
    ) as pool:
        results = list(tqdm(
            pool.imap_unordered(process_single_image, tasks, chunksize=chunksize),
            total=len(tasks),
            desc='  Selective Search'
        ))

    for img_id, boxes in results:
        proposals_dict[img_id] = boxes

    elapsed = time.time() - t0

    with open(cache_path, 'wb') as f:
        pickle.dump(proposals_dict, f)

    lens = [len(v) for v in proposals_dict.values()]
    print(f'  Đã lưu → {cache_path}')
    print(f'  Thời gian : {elapsed:.1f}s ({elapsed/len(img_ids):.3f}s/ảnh)')
    print(f'  Proposals : avg={int(sum(lens)/len(lens))}, min={min(lens)}, max={max(lens)}')
    return proposals_dict


print(f'Master function OK — sẽ chạy với NUM_CORES = {NUM_CORES}')

## 7. Tính proposals — Tập Train

In [ ]:
print('=== TRAIN SET ===')
train_proposals = precompute_ss_proposals(
    img_dir    = TRAIN_IMG,
    ann_file   = TRAIN_ANN,
    cache_path = TRAIN_SS_CACHE,
    max_props  = MAX_PROPOSALS,
    num_cores  = NUM_CORES,
    overwrite  = OVERWRITE,
)

## 8. Tính proposals — Tập Validation

In [ ]:
print('=== VALIDATION SET ===')
valid_proposals = precompute_ss_proposals(
    img_dir    = VALID_IMG,
    ann_file   = VALID_ANN,
    cache_path = VALID_SS_CACHE,
    max_props  = MAX_PROPOSALS,
    num_cores  = NUM_CORES,
    overwrite  = OVERWRITE,
)

## 9. Tính proposals — Tập Test + Thống kê kết quả

In [ ]:
import matplotlib.ticker as ticker

print('=== TEST SET ===')
test_proposals = precompute_ss_proposals(
    img_dir    = TEST_IMG,
    ann_file   = TEST_ANN,
    cache_path = TEST_SS_CACHE,
    max_props  = MAX_PROPOSALS,
    num_cores  = NUM_CORES,
    overwrite  = OVERWRITE,
)

# ── Bảng thống kê tổng hợp ────────────────────────────────────────────────────
print('\n===== THỐNG KÊ KẾT QUẢ =====')
print(f'{"Tập":>12} {"Số ảnh":>8} {"Avg props":>10} {"Min":>6} {"Max":>6} {"File size":>10}')
print('-' * 58)

split_data = [
    ('Train',      train_proposals, TRAIN_SS_CACHE),
    ('Validation', valid_proposals, VALID_SS_CACHE),
    ('Test',       test_proposals,  TEST_SS_CACHE),
]
for name, props, path in split_data:
    lens  = [len(v) for v in props.values()]
    fsize = os.path.getsize(path) / 1024**2 if os.path.exists(path) else 0
    print(f'{name:>12} {len(props):>8} {int(sum(lens)/len(lens)):>10}'
          f' {min(lens):>6} {max(lens):>6} {fsize:>8.1f} MB')

# ── Biểu đồ phân phối proposals ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Phân phối số lượng Selective Search Proposals / ảnh',
             fontsize=13, fontweight='bold')

for ax, (name, props, _) in zip(axes, split_data):
    counts = [len(v) for v in props.values()]
    avg    = sum(counts) / len(counts)
    ax.hist(counts, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(avg, color='red', ls='--', label=f'Avg: {int(avg)}')
    ax.set_title(f'{name} ({len(props)} ảnh)', fontweight='bold')
    ax.set_xlabel('Số proposals / ảnh')
    ax.set_ylabel('Số ảnh')
    ax.legend(fontsize=9)
    ax.grid(True, ls='--', alpha=0.4)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(200))

plt.tight_layout()
fig.savefig(os.path.join(DATASET_DIR, 'ss_proposals_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Kiểm tra mẫu ──────────────────────────────────────────────────────────────
print('\n===== KIỂM TRA MẪU =====')
for name, props, _ in split_data:
    sid = list(props.keys())[0]
    print(f'{name}: img_id={sid} → {len(props[sid])} proposals | ví dụ: {props[sid][:2]}')

print(f'\nHoàn thành! {len(split_data)} file .pkl sẵn sàng cho huấn luyện Fast R-CNN.')